# Model Deployment Using FastAPI

In this practice you will analyse which customer segments responded to a skin clinic's marketing campaign, turn the analysis into a FastAPI application with a **/campaign-analysis** endpoint that displays the results as tables, and deploy it on Render.

Everything here uses only what Lesson 2 covers: creating a FastAPI app, defining GET endpoints, returning HTML responses, running with Uvicorn, and deploying to Render.

## Background

A skin clinic ran a marketing campaign targeting 10,000 customers. The clinic wants to know which customer segments are more likely to respond, based on demographics and purchase behaviour. The insights will help it target future campaigns and increase campaign effectiveness.

**Before you start:** install the libraries with `pip install fastapi uvicorn httpx pandas`. `httpx` is needed only for testing the app inside this notebook.

## Data Description

This practice uses **skin clinic campaign.csv** (10,000 customers, one row per customer), included in the download.

| Column Name | Description |
|---|---|
| CustID | Unique customer identifier |
| Gender | Female or Male |
| AgeGroup | <30, 30-50 or >50 |
| Purchase_Last_Quarter | Whether the customer made a purchase in the last quarter (Yes/No) |
| Unique_Products_Purchased | Number of different products bought in the last year (1 to 15) |
| Response_to_Campaign | **Target variable:** whether the customer responded to the campaign (Yes/No) |

In [1]:
import pandas as pd

df = pd.read_csv("skin clinic campaign.csv")
print(df.shape)
df.head()

(10000, 6)


,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes


## 1. Load and Prepare the Data

We convert **Response_to_Campaign** to 1 (Yes) and 0 (No). The mean of this column is then the response rate. We also group **Unique_Products_Purchased** into three bands: 1-4, 5-8 and more than 8.

In [2]:
df["Response"] = df["Response_to_Campaign"].map({"Yes": 1, "No": 0})

def categorize_products(x):
    if x <= 4:
        return "1-4"
    elif x <= 8:
        return "5-8"
    else:
        return ">8"

df["Product_Category"] = df["Unique_Products_Purchased"].apply(categorize_products)

print("Overall response rate:", round(df["Response"].mean() * 100, 1), "%")

Overall response rate: 39.9 %


## 2. Response Rate by Segment

A helper function returns the response rate (%) for each level of a column. We use it for the four tables the clinic asked for: gender, age group, purchase in the last quarter, and product usage.

In [3]:
def response_table(group_col):
    result = df.groupby(group_col)["Response"].mean().reset_index()
    result["Response Rate (%)"] = (result["Response"] * 100).round(2)
    return result.drop(columns=["Response"])

for col in ["Gender", "AgeGroup", "Purchase_Last_Quarter", "Product_Category"]:
    display(response_table(col))

,Gender,Response Rate (%)
0,Female,43.77
1,Male,34.08


,AgeGroup,Response Rate (%)
0,30-50,47.06
1,<30,32.93
2,>50,32.60


,Purchase_Last_Quarter,Response Rate (%)
0,No,21.69
1,Yes,49.65


,Product_Category,Response Rate (%)
0,1-4,18.08
1,5-8,38.48
2,>8,52.00


**Interpretation:** the overall response rate is about 40%. Customers aged 30-50, those who bought in the last quarter, and those who buy more than 8 different products respond best (around 47-52%). Customers who bought 1-4 products respond least (about 18%). Future campaigns should prioritise recent, multi-product customers in the 30-50 age group.

## 3. Build the FastAPI Application

We create a FastAPI app with a GET endpoint **/campaign-analysis**. `response_class=HTMLResponse` tells FastAPI to send the page as HTML rather than JSON, so the four tables display in the browser. pandas `to_html()` converts each table to HTML.

In [4]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

app = FastAPI()

@app.get("/campaign-analysis", response_class=HTMLResponse)
def campaign_analysis():
    gender = response_table("Gender")
    age = response_table("AgeGroup")
    purchase = response_table("Purchase_Last_Quarter")
    product = response_table("Product_Category")

    return f"""
    <html>
    <body style="font-family: Arial;">
        <h1>Campaign Analysis</h1>
        <h2>Gender vs Campaign Response</h2>
        {gender.to_html(index=False)}
        <h2>Age Group vs Campaign Response</h2>
        {age.to_html(index=False)}
        <h2>Purchase Last Quarter vs Campaign Response</h2>
        {purchase.to_html(index=False)}
        <h2>Product Category vs Campaign Response</h2>
        {product.to_html(index=False)}
    </body>
    </html>
    """

## 4. Test the Endpoint

FastAPI's `TestClient` lets us call the endpoint from the notebook without starting a server. A status code of 200 means the request succeeded, and the content type confirms the response is HTML.

In [5]:
from fastapi.testclient import TestClient
from IPython.display import HTML

client = TestClient(app)
response = client.get("/campaign-analysis")

print("Status code:", response.status_code)
print("Content type:", response.headers["content-type"])
HTML(response.text)

Status code: 200
Content type: text/html; charset=utf-8


Gender,Response Rate (%)
Female,43.77
Male,34.08
AgeGroup,Response Rate (%)
30-50,47.06
<30,32.93
>50,32.60
Purchase_Last_Quarter,Response Rate (%)
No,21.69
Yes,49.65
Product_Category,Response Rate (%)


## 5. Save the App as main.py

To run and deploy the app, the code must be in a Python file. We create a project folder, **campaign-api**, copy the dataset into it, and write **main.py** and **requirements.txt** there. Keeping the app in its own folder means it does not overwrite the lesson's own example files.

In [6]:
from pathlib import Path
import shutil

project = Path("campaign-api")
project.mkdir(exist_ok=True)
shutil.copy("skin clinic campaign.csv", project)
print("Project folder ready:", sorted(f.name for f in project.iterdir()))

Project folder ready: ['skin clinic campaign.csv']


In [7]:
%%writefile campaign-api/main.py
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import pandas as pd

app = FastAPI()

# Load the data
df = pd.read_csv("skin clinic campaign.csv")

# Convert the response to 1 (Yes) and 0 (No)
df["Response"] = df["Response_to_Campaign"].map({"Yes": 1, "No": 0})

# Group the number of unique products purchased
def categorize_products(x):
    if x <= 4:
        return "1-4"
    elif x <= 8:
        return "5-8"
    else:
        return ">8"

df["Product_Category"] = df["Unique_Products_Purchased"].apply(categorize_products)

# Response rate (%) for each level of a column
def response_table(group_col):
    result = df.groupby(group_col)["Response"].mean().reset_index()
    result["Response Rate (%)"] = (result["Response"] * 100).round(2)
    return result.drop(columns=["Response"])


@app.get("/campaign-analysis", response_class=HTMLResponse)
def campaign_analysis():
    gender = response_table("Gender")
    age = response_table("AgeGroup")
    purchase = response_table("Purchase_Last_Quarter")
    product = response_table("Product_Category")

    return f"""
    <html>
    <body style="font-family: Arial;">
        <h1>Campaign Analysis</h1>
        <h2>Gender vs Campaign Response</h2>
        {gender.to_html(index=False)}
        <h2>Age Group vs Campaign Response</h2>
        {age.to_html(index=False)}
        <h2>Purchase Last Quarter vs Campaign Response</h2>
        {purchase.to_html(index=False)}
        <h2>Product Category vs Campaign Response</h2>
        {product.to_html(index=False)}
    </body>
    </html>
    """

Writing campaign-api/main.py


In [8]:
%%writefile campaign-api/requirements.txt
fastapi
uvicorn
pandas

Writing campaign-api/requirements.txt


## 6. Run the App Locally

Open a terminal in the **campaign-api** folder, then run:

```bash
uvicorn main:app --reload
```

Open `http://127.0.0.1:8000/campaign-analysis` in your browser to see the tables. FastAPI's automatic documentation is at `http://127.0.0.1:8000/docs`. Press **Ctrl + C** in the terminal to stop the server.

## 7. Deploy on Render

1. Put the contents of the **campaign-api** folder (**main.py**, **requirements.txt** and **skin clinic campaign.csv**) in a GitHub repository.
2. On Render, create a new **Web Service** from that repository.
3. Build command: `pip install -r requirements.txt`
4. Start command: `uvicorn main:app --host 0.0.0.0 --port $PORT`
5. When the deploy finishes, open `https://<your-app-name>.onrender.com/campaign-analysis` and check that the four tables display.

## Summary

- A FastAPI app exposes the analysis through a GET endpoint, **/campaign-analysis**.
- `HTMLResponse` and pandas `to_html()` display the results as tables in the browser.
- `TestClient` tests an endpoint without starting a server; Uvicorn runs it locally.
- Render deploys the app from a GitHub repository with a build command and a start command, giving a public link.
- The analysis shows that recent, multi-product customers aged 30-50 respond best to the campaign.